# Model Development
The objective of this notebook is to develop an image-classification model for the Paddy Disease Classification
dataset using EfficientNetV2-S with pretrained ImageNet weights. The main steps are:

- Load the pretrained EfficientNetV2-S model.
- Inspect its architecture.
- Replace the original classification head for the 10 Paddy Disease classes.
- Establish the initial transfer-learning strategy.
- Identify the trainable and frozen parameters.
- Define the loss function and optimizer.
- Perform a forward-pass sanity check.

### Transfer-Learning Strategy

We will initially use EfficientNetV2-S as a pretrained feature extractor and
train a new classification head for the ten Paddy Disease classes. 
After establishing this baseline, we will fine-tune the pretrained network.

## 1. Imports, Device, Project Setup
We define the libraries, project paths, and computation device used throughout
this notebook.

The notebook is designed to be run independently of the previous notebooks.

### 1.1: Import Libraries and Project Setup

In [1]:
import torch
from torch import nn
from torchvision.models import (
    efficientnet_v2_s,
    EfficientNet_V2_S_Weights
)

from pathlib import Path
import sys

# project root
PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# project modules
from src.data import train_loader, val_loader, NUM_CLASSES, CLASS_NAMES

Training samples: 8325
Validation samples: 2082


### 1.2: Choose Device

In [2]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(f"Using device: {device}")

Using device: mps


## 2. Load the Pretrained EfficientNetV2-S
We load EfficientNetV2-S with the ImageNet-1K pretrained weights selected in
the data-preparation notebook.

At this stage, we keep the original architecture and classifier unchanged so
that we can inspect what the pretrained model provides before adapting it to
the Paddy Disease Classification task.

In [3]:
weights = EfficientNet_V2_S_Weights.DEFAULT

model = efficientnet_v2_s(weights=weights)

print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, bias=T

### 2.1: Inspect The Classifier

In [4]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [5]:
print(f"Input features: {model.classifier[1].in_features}")
print(f"Output classes: {model.classifier[1].out_features}")

Input features: 1280
Output classes: 1000


### 2.2 Replace the ImageNet Classifier
The pretrained EfficientNetV2-S classifier is designed for the 1000 ImageNet
classes. Our task contains 10 Paddy Disease classes. We therefore replace only the final linear layer, preserving the pretrained feature-extraction network.

In [6]:
model.classifier[1] = nn.Linear(
    in_features=model.classifier[1].in_features,
    out_features=NUM_CLASSES
)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=10, bias=True)
)


## 3. Baseline Model

### 3.1 Freeze the Pretrained Backbone
For the initial transfer-learning experiment, we freeze the pretrained
EfficientNetV2-S feature extractor and train only the newly initialized
classification head. This allows us to first evaluate how useful the ImageNet-learned
representation is for the Paddy Disease Classification task before
fine-tuning the pretrained features.

In [7]:
for parameter in model.features.parameters():
    parameter.requires_grad = False

### 3.2 Inspect Trainable Parameters
After freezing the pretrained feature extractor, we verify which model
parameters remain trainable.
For the baseline, only the newly replaced classification head should have
`requires_grad=True`.

In [8]:
trainable_parameters = [
    name for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

print("Trainable parameters:")
for name in trainable_parameters:
    print(name)

Trainable parameters:
classifier.1.weight
classifier.1.bias


### 3.3 Define the Loss Function
We use cross-entropy loss for the ten-class classification task.

The classifier produces one logit for each Paddy Disease class, and
`CrossEntropyLoss` combines the softmax operation with the negative
log-likelihood calculation internally.

In [9]:
criterion = nn.CrossEntropyLoss()

### 3.4 Define the Optimizer
Since only the newly initialized classification head is trainable, the
optimizer is applied to those parameters.

We use AdamW with a learning rate of `1e-3` as the initial baseline.

In [10]:
optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=1e-3
)

### 3.5 Forward-Pass Sanity Check
Before training, we verify that a batch of images can pass through the model
and produce outputs with the expected shape.

The model is temporarily placed in evaluation mode so that dropout and
batch-normalization behave deterministically during this sanity check.

In [11]:
model = model.to(device)
model.eval()

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    logits = model(images)

print(f"Input shape: {images.shape}")
print(f"Output shape: {logits.shape}")
print(f"Labels shape: {labels.shape}")

Input shape: torch.Size([32, 3, 384, 384])
Output shape: torch.Size([32, 10])
Labels shape: torch.Size([32])


### 3.6: Baseline Training
We train the baseline model using the frozen EfficientNetV2-S feature extractor
and the newly initialized ten-class classification head.

The model is trained on the training set and evaluated on the validation set
after each epoch.

In [12]:
NUM_EPOCHS = 5

In [13]:
for epoch in range(NUM_EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(
        f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch [1/5] Train Loss: 1.6841 Val Loss: 1.4093
Epoch [2/5] Train Loss: 1.3762 Val Loss: 1.2439
Epoch [3/5] Train Loss: 1.2671 Val Loss: 1.1781
Epoch [4/5] Train Loss: 1.2134 Val Loss: 1.1330
Epoch [5/5] Train Loss: 1.1713 Val Loss: 1.1025


### 3.7 Evaluate the Baseline Model
We evaluate the trained baseline model on the validation set using accuracy and
macro F1 score.

Because the dataset is imbalanced, macro F1 is particularly useful: it gives
each class equal importance rather than allowing the larger classes to
dominate the overall metric.

We also construct a confusion matrix so that performance across individual
classes can be examined.

In [14]:
model.eval()

correct = 0
total = 0

confusion_matrix = torch.zeros(
    NUM_CLASSES,
    NUM_CLASSES,
    dtype=torch.int64
)

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        predictions = logits.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        for true_label, predicted_label in zip(labels, predictions):
            confusion_matrix[true_label, predicted_label] += 1

accuracy = correct / total

true_positives = confusion_matrix.diag()
false_positives = confusion_matrix.sum(dim=0) - true_positives
false_negatives = confusion_matrix.sum(dim=1) - true_positives

precision = true_positives / (
    true_positives + false_positives
).clamp(min=1)

recall = true_positives / (
    true_positives + false_negatives
).clamp(min=1)

f1 = 2 * precision * recall / (
    precision + recall
).clamp(min=1e-8)

macro_f1 = f1.mean().item()

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation Macro F1: {macro_f1:.4f}")

Validation Accuracy: 0.6402
Validation Macro F1: 0.6008


### 3.8 Per-Class Performance
The overall accuracy and macro F1 score can hide differences between
individual classes. We therefore examine the F1 score for each Paddy Disease
class.

This is particularly important because the dataset is imbalanced.

In [15]:
for class_name, class_f1 in zip(CLASS_NAMES, f1):
    print(f"{class_name:<28} F1: {class_f1.item():.4f}")

bacterial_leaf_blight        F1: 0.3206
bacterial_leaf_streak        F1: 0.6000
bacterial_panicle_blight     F1: 0.6980
blast                        F1: 0.6649
brown_spot                   F1: 0.5258
dead_heart                   F1: 0.8109
downy_mildew                 F1: 0.4833
hispa                        F1: 0.5916
normal                       F1: 0.7393
tungro                       F1: 0.5734


### 3.9: Save Baseline Metrics

In [16]:
import json

metrics_dir = PROJECT_ROOT / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

baseline_metrics = {
    "model": "EfficientNetV2-S",
    "pretrained": True,
    "backbone_frozen": True,
    "epochs": NUM_EPOCHS,
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "per_class_f1": {
        class_name: class_f1.item()
        for class_name, class_f1 in zip(CLASS_NAMES, f1)
    }
}

metrics_path = metrics_dir / "baseline.json"

with open(metrics_path, "w") as file:
    json.dump(baseline_metrics, file, indent=4)

print(f"Baseline metrics saved to: {metrics_path}")

Baseline metrics saved to: /Users/amitkumar/Desktop/Computing/Projects/paddy-disease-classification/results/metrics/baseline.json
